# Seminário 2: Geração e exploração de um banco de dados acerca dos docentes do ICMC-USP

## Autores

| Nome                                      | nUSP     |
| :---------------------------------------- | :------- |
| Lucas de Oliveira Ferreira                | 13695042 |
| Guilherme de Abreu Barreto                | 12543033 |
| Jhonathan Oliveira Alves                  | 11838116 |
| Lucas Pereira Franco de Almeida           | 12675020 |
| Miguel Prates Ferreira de Lima Cantanhede | 13672745 |


## Descrição

Neste notebook encontram-se todas as visualizações descritas em [nosso relatório](https://github.com/de-abreu/visualizacao_computacional/blob/main/seminario2/README.md), de tal forma que o leitor possa interatir com as mesmas. Para mais informações sobre as técnicas de visualização ou o contexto em que estas estão sendo empregadas, recomenda-se a leitura deste documento.

Para sua melhor visualização recomenda-se, após executar as células deste notebook, acessar as visualizações abrindo uma nova aba do navegador para as páginas web em que estas foram geradas. Os links para acessar estas encontram-se listadas abaixo:

- [Diagrama de arcos](http://127.0.0.1:8051/)
- [Gráfico de linhas](http://127.0.0.1:8052/)

Fizemos uso de três visualizações no total, descritas abaixo, cada qual para responder a dadas perguntas específicas:

### Diagrama de arcos

Quais projetos de pesquisa ou artigos científicos foram resultados da colaboração entre os pesquisadores do ICMC e,

- quais os professores mais colaborativos?
- com quem estes colaboram?
- quantas vezes estes já colaboraram?

### Gráfico de bolhas

Como grupos de pesquisa diferentes do ICMC se relacionam quando análisamos os títulos dos artigos publicados por seus membros docentes e,

- quais os grupos com mais e menos artigos?
- quais grupos possuem artigos que tratam de temas/áreas parecidas?


### Gráfico de linhas

Qual a produtividade dos docentes em função do tempo e,

- como esta se compara aos demais docentes deste mesmo instituto?
- quais são os artigos, por ano, de cada professor?


## Dependências

In [3]:
from sqlalchemy import create_engine
import pandas as pd


engine = create_engine("sqlite:///database/lattes.db") # Acesso ao banco de dados

## Diagrama de Arcos

In [4]:
from arc_diagram.collab_dashboard import create_collab_dashboard

collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""

# Load dataframe with data extracted from the database query
try:
    collaborations: pd.DataFrame = pd.read_sql_query(collaborations_query, engine)
except Exception as e:
    print(f"✗ Erro ao carregar dados de colaboração do banco de dados: {e}")
    print(
        "  - Verifique se o banco de dados existe e contém as tabelas necessárias"
    )
    raise

print("✓ Dados de colaboração carregados com sucesso")
print(f"  - {len(collaborations)} registros carregados")

# Create and run the Dash app for the Arc Diagram visualization
app = create_collab_dashboard(
    collab_df=collaborations,
    title="Colaborações entre Professores do ICMC, em artigos e projetos de pesquisa",
    legend_title="Colaborações",
)
app.run(host="127.0.0.1", port=8051)

✓ Dados de colaboração carregados com sucesso
  - 807 registros carregados


## Gráfico de Bolhas

In [4]:
'''from bubble_plot.bubble_plot import *

arts = load_articles(engine)
groups = load_groups()
macros_faltantes = load_macros_faltantes()

emb_articles, emb_groups, emb_macros = embeddings(arts, groups, macros_faltantes)

arts = get_groups(arts, emb_articles, groups, emb_groups)

XY = dim_reduction(emb_articles)
arts["x"], arts["y"] = XY[:, 0], XY[:, 1]

arts = map_macro(arts, emb_articles, macros_faltantes, emb_macros)

bubbles = create_bubble(arts)
#plotting_bubbles(bubbles)

cols = ["x", "y", "n", "macro", "grupo"]
bubbles[cols].to_json("bubble_plot/bubbles.json", orient="records")
print("Salvei bubbles.json")'''


Artigos: 5027 | Professores: 126


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

/home/lucas_of/anaconda3/envs/VisComp12/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Salvei bubbles.json


In [1]:
import os
import threading
import webbrowser
from http.server import HTTPServer, SimpleHTTPRequestHandler

PORT = 8000

def run_server():
    httpd = HTTPServer(("localhost", PORT), SimpleHTTPRequestHandler)
    print(f"Servidor rodando em http://localhost:{PORT}/bubble_plot/fisheye.html")
    httpd.serve_forever()

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

webbrowser.open(f"http://localhost:{PORT}/bubble_plot/fisheye.html")


Servidor rodando em http://localhost:8000/bubble_plot/fisheye.html


True

gio: http://localhost:8000/bubble_plot/fisheye.html: Operation not supported


## Gráfico de Linhas

Para visualizar as duas colunas, utilizamos um código disponível no [stackoverflow](https://stackoverflow.com/questions/38783027/jupyter-notebook-display-two-pandas-tables-side-by-side)

In [5]:
from IPython.display import display_html
from itertools import chain,cycle
def display_side_by_side(*args,titles=cycle([''])):
    html_str=''
    for df,title in zip(args, chain(titles,cycle(['</br>'])) ):
        html_str+='<th style="text-align:center"><td style="vertical-align:top">'
        html_str+=f'<h2 style="text-align: center;">{title}</h2>'
        html_str+=df.to_html().replace('table','table style="display:inline"')
        html_str+='</td></th>'
    display_html(html_str,raw=True)

In [6]:
from line_graph.data import get_data
df,df_counts = get_data()


display_side_by_side(df.head(),df_counts.head(), titles=['df','df_counts']) #we left 3rd empty...

,researcher_name,article_title,article_year
0,Francisco Louzada Neto,Efficient closed-form maximum a posteriori estimators for the gamma distribution,2018
1,Alexandre Cláudio Botazzo Delbem,Ant-Based Phylogenetic Reconstruction (ABPR): A new distance algorithm for phylogenetic estimation based on ant colony optimization,2008
2,Reiko Aoki,Asymptotic Efficiency of Method of Moments Estimators under Null Intercept Measurement Error Regression Models,2002
3,Miguel Vinicius Santini Frasson,Measure Neutral Functional Differential Equations as Generalized ODEs,2019
4,André Carlos Ponce de Leon Ferreira de Carvalho,Cluster ensemble selection based on relative validity indexes,2013
,researcher_name,year,count_articles
0,Adenilso da Silva Simão,2002,2
1,Adenilso da Silva Simão,2003,1
2,Adenilso da Silva Simão,2006,1
3,Adenilso da Silva Simão,2008,3


In [7]:
from line_graph.dashboard import create_visualization

app = create_visualization(df,df_counts)

PORT = 8052
HOST = 'localhost'
print(f"{HOST}:{PORT}")
app.run(host=HOST, port=PORT)

localhost:8052


In [1]:
# Inicia o app combinado (Diagrama de Arcos + Gráfico de Linhas sincronizados)
from combined_dashboard import create_combined_dashboard
from line_graph.data import get_data
import sqlite3, pandas as pd

# carrega dados necessários
df, df_counts = get_data()
conn = sqlite3.connect('database/lattes.db')
collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""
collaborations = pd.read_sql_query(collaborations_query, conn)

app = create_combined_dashboard(collaborations, df, df_counts)
PORT = 8060
HOST = 'localhost'
print(f"Starting combined dashboard at {HOST}:{PORT}")
app.run(host=HOST, port=PORT)

Starting combined dashboard at localhost:8060
